# LCC — Corrected Paper Experiment: E1–E5
Based on **UpdatedRevise_LCC_Hybrid_IF_RBF_SVM_F.docx.pdf**, Tables 6–7 and Sections 2.6–2.9. This revision replaces the earlier three-model design. All outputs are cleared: these five experiments must be run before results can be claimed.

| ID | Inputs | Classifier | SMOTE |
|---|---|---|---|
| E1 | Nine RF-selected features | Isolation Forest | No |
| E2 | Nine features | RBF-SVM | No |
| E3 | Nine features | RBF-SVM | Yes |
| E4 | Nine features + normalized IF score | RBF-SVM | No |
| E5 | Nine features + normalized IF score | RBF-SVM | Yes |

Primary comparison: **E1 vs E5**. IF-score contribution: E2 vs E4 and E3 vs E5. SMOTE contribution: E2 vs E3 and E4 vs E5. McNemar test: E2 vs E4, alpha 0.05.

**Paper inconsistencies resolved explicitly:** Table 6/Section 2.7 and the objectives specify SMOTE for E5, although Section 2.4 retains older wording. Partition counts vary between 175,341/82,332 and 175,332/82,422. This notebook reports actual counts without changing labels or manufacturing records. A discrepancy must be reconciled in the manuscript before claiming exact reproduction.

Preprocessing, IF-score generation and SMOTE are fitted within each supervised CV training fold. Validation and test are never resampled. E1 is fitted separately on the official training file to reproduce the reference baseline. The supervised training subsample is shared across E2–E5.

**Run cost:** four searches × 16 candidates × five folds = 320 CV fits, plus refits. Default supervised cap is 40,000 original rows; SMOTE can increase that count. Set the cap before seeing test results and document it. Do not compare timings across different hardware as if equivalent.


In [ ]:
# Run once if dependencies are missing, then restart the kernel if requested:
# %pip install numpy pandas scikit-learn imbalanced-learn scipy matplotlib joblib
import os, sys, math, time, json, hashlib, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn, imblearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, matthews_corrcoef, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve, auc)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from scipy.stats import binomtest
RANDOM_STATE = 42
EULER_GAMMA = 0.5772156649015329
FEATURES = ['sttl','ct_state_ttl','dload','rate','dmean','dttl','ackdat','sload','sbytes']
VALIDATION_SIZE = 0.20
SVM_MAX_TRAIN_ROWS = 40000  # None uses all supervised training rows
CV_FOLDS = 5
N_JOBS = 1  # Avoid concurrent large SVC kernel caches
PARAM_GRID = {'svc__C':[0.1,1,10,100], 'svc__gamma':['scale',0.01,0.1,1]}
# CV scoring is an explicit implementation choice; validation threshold uses F0.5.
CV_SCORING = 'f1'
OUTPUT = Path('lcc_corrected_outputs'); OUTPUT.mkdir(exist_ok=True)
TRAIN_PATH = None  # Set an explicit CSV path, or auto-discover under /kaggle/input
TEST_PATH = None


In [ ]:
def locate(filename):
    matches = sorted(Path('/kaggle/input').rglob(filename))
    if len(matches) != 1:
        raise ValueError(f'Set an explicit path: found {len(matches)} matches for {filename}')
    return matches[0]

def read_partition(path):
    import csv
    with open(path, encoding='utf-8-sig', newline='') as stream:
        header = [v.strip().lower() for v in next(csv.reader(stream))]
    if len(header) != len(set(header)):
        raise ValueError('Duplicate CSV headers')
    frame = pd.read_csv(path)
    frame.columns = frame.columns.astype(str).str.strip().str.lower()
    missing = set(FEATURES + ['label']) - set(frame.columns)
    if missing or frame.empty:
        raise ValueError(f'Empty partition or missing columns: {sorted(missing)}')
    y = pd.to_numeric(frame.label, errors='coerce')
    if not y.isin([0,1]).all() or y.nunique()!=2:
        raise ValueError('Partition must contain valid Normal (0) and Attack (1) labels')
    X = frame[FEATURES].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan)
    return X, y.to_numpy(dtype=int)

TRAIN_PATH = Path(TRAIN_PATH) if TRAIN_PATH else locate('UNSW_NB15_training-set.csv')
TEST_PATH = Path(TEST_PATH) if TEST_PATH else locate('UNSW_NB15_testing-set.csv')
if TRAIN_PATH.resolve() == TEST_PATH.resolve(): raise ValueError('Train and test paths must differ')
X_full, y_full = read_partition(TRAIN_PATH)
X_test, y_test = read_partition(TEST_PATH)
if X_full.isna().all().any(): raise ValueError('Training contains an entirely missing feature')
counts = pd.DataFrame([
    dict(partition=name, rows=len(y), normal=int((y==0).sum()), attack=int((y==1).sum()))
    for name,y in [('training',y_full),('testing',y_test)]])
display(counts)
if (len(y_full),len(y_test)) != (175341,82332):
    warnings.warn('Actual partition sizes differ from the commonly used official split. Verify provenance.')
print('Manuscript alternative counts: 175332 training / 82422 test. No rows were altered to match.')
indices = np.arange(len(y_full))
fit_idx, val_idx = train_test_split(indices, test_size=VALIDATION_SIZE,
    stratify=y_full, random_state=RANDOM_STATE)
if SVM_MAX_TRAIN_ROWS and len(fit_idx)>SVM_MAX_TRAIN_ROWS:
    fit_idx,_ = train_test_split(fit_idx, train_size=SVM_MAX_TRAIN_ROWS,
        stratify=y_full[fit_idx], random_state=RANDOM_STATE)
X_fit, y_fit = X_full.iloc[fit_idx], y_full[fit_idx]
X_val, y_val = X_full.iloc[val_idx], y_full[val_idx]
print('Supervised original rows:',len(y_fit),'validation rows:',len(y_val))


In [ ]:
def c_factor(n):
    n = int(n)
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    return 2.0 * (np.log(n - 1) + EULER_GAMMA) - 2.0 * (n - 1) / n

class IsolationNode:
    __slots__ = ('feature', 'threshold', 'left', 'right', 'size', 'is_leaf')

    def __init__(self, feature=None, threshold=None, left=None, right=None, size=0, is_leaf=False):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.size = int(size)
        self.is_leaf = bool(is_leaf)

class IsolationTree:

    def __init__(self, height_limit, rng):
        self.height_limit = int(height_limit)
        self.rng = rng
        self.root = None

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        self.root = self._build(X, 0)
        return self

    def _build(self, X, height):
        n_samples = X.shape[0]
        if height >= self.height_limit or n_samples <= 1:
            return IsolationNode(size=n_samples, is_leaf=True)
        minimums = np.min(X, axis=0)
        maximums = np.max(X, axis=0)
        valid_features = np.where(maximums > minimums)[0]
        if len(valid_features) == 0:
            return IsolationNode(size=n_samples, is_leaf=True)
        feature = int(self.rng.choice(valid_features))
        threshold = float(self.rng.uniform(minimums[feature], maximums[feature]))
        left_mask = X[:, feature] < threshold
        right_mask = ~left_mask
        if left_mask.sum() == 0 or right_mask.sum() == 0:
            return IsolationNode(size=n_samples, is_leaf=True)
        left = self._build(X[left_mask], height + 1)
        right = self._build(X[right_mask], height + 1)
        return IsolationNode(feature=feature, threshold=threshold, left=left, right=right, size=n_samples, is_leaf=False)

    def path_length(self, x):
        node = self.root
        depth = 0
        while not node.is_leaf:
            if x[node.feature] < node.threshold:
                node = node.left
            else:
                node = node.right
            depth += 1
        if node.size > 1:
            return float(depth) + c_factor(node.size)
        return float(depth)

class CustomIsolationForest:

    def __init__(self, n_trees=100, sample_size=256, random_state=42):
        self.n_trees = int(n_trees)
        self.sample_size = int(sample_size)
        self.random_state = int(random_state)
        self.trees = []
        self.actual_sample_size = None
        self.height_limit = None

    def fit(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_samples = X.shape[0]
        self.actual_sample_size = min(self.sample_size, n_samples)
        self.height_limit = int(math.ceil(math.log2(self.actual_sample_size)))
        master_rng = np.random.default_rng(self.random_state)
        self.trees = []
        for _ in range(self.n_trees):
            sample_indices = master_rng.choice(n_samples, size=self.actual_sample_size, replace=False)
            X_sample = X[sample_indices]
            tree_seed = int(master_rng.integers(0, np.iinfo(np.int32).max))
            tree = IsolationTree(height_limit=self.height_limit, rng=np.random.default_rng(tree_seed))
            tree.fit(X_sample)
            self.trees.append(tree)
        return self

    def mean_path_length(self, X):
        X = np.asarray(X, dtype=np.float64)
        total = np.zeros(len(X))
        for tree in self.trees:
            stack = [(tree.root, np.arange(len(X)), 0)]
            while stack:
                node, rows, depth = stack.pop()
                if not len(rows):
                    continue
                if node.is_leaf:
                    total[rows] += depth + c_factor(node.size)
                else:
                    left = X[rows, node.feature] < node.threshold
                    stack.append((node.left, rows[left], depth + 1))
                    stack.append((node.right, rows[~left], depth + 1))
        return total / len(self.trees)

    def anomaly_score(self, X):
        paths = self.mean_path_length(X)
        normalizer = c_factor(self.actual_sample_size)
        return np.power(2.0, -(paths / normalizer))

In [ ]:
class IFScoreAugmenter(TransformerMixin, BaseEstimator):
    """Train-only IF and score normalization; cloned inside each CV fold."""
    def __init__(self, n_trees=100, sample_size=256, random_state=42):
        self.n_trees=n_trees; self.sample_size=sample_size; self.random_state=random_state
    def fit(self, X, y=None):
        self.n_features_in_=np.asarray(X).shape[1]
        self.if_scaler_=MinMaxScaler().fit(X)
        values=self.if_scaler_.transform(X)
        self.forest_=CustomIsolationForest(self.n_trees,self.sample_size,self.random_state).fit(values)
        self.score_scaler_=MinMaxScaler(clip=True).fit(self.forest_.anomaly_score(values).reshape(-1,1))
        return self
    def transform(self, X):
        values=np.asarray(X,dtype=float)
        scores=self.forest_.anomaly_score(self.if_scaler_.transform(values))
        normalized=self.score_scaler_.transform(scores.reshape(-1,1))
        return np.column_stack([values,normalized])

def make_model(augmented, smote):
    steps=[('imputer',SimpleImputer(strategy='median',keep_empty_features=True))]
    if augmented: steps.append(('augment',IFScoreAugmenter()))
    steps.append(('standardize',StandardScaler()))
    if smote: steps.append(('smote',SMOTE(random_state=RANDOM_STATE,k_neighbors=5)))
    steps.append(('svc',SVC(kernel='rbf',cache_size=512)))
    return Pipeline(steps, memory=str(OUTPUT/'cv_cache'))

def select_threshold(y, scores):
    precision,recall,thresholds=precision_recall_curve(y,scores)
    denominator=.25*precision[:-1]+recall[:-1]
    f05=np.divide(1.25*precision[:-1]*recall[:-1],denominator,
                  out=np.zeros_like(denominator),where=denominator!=0)
    # Include the all-negative candidate for completeness; ties choose lowest threshold.
    candidates=np.r_[thresholds,np.nextafter(np.max(scores),np.inf)]
    objective=np.r_[f05,0.0]
    i=int(np.argmax(objective))
    return float(candidates[i]),float(objective[i])

def evaluate(y,pred,scores):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    precision,recall,_=precision_recall_curve(y,scores)
    return dict(accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),
        f1=float(f1_score(y,pred,zero_division=0)),
        f0_5=float(fbeta_score(y,pred,beta=.5,zero_division=0)),
        mcc=float(matthews_corrcoef(y,pred)),roc_auc=float(roc_auc_score(y,scores)),
        pr_auc=float(auc(recall,precision)),average_precision=float(average_precision_score(y,scores)),
        specificity=float(tn/(tn+fp)),fpr=float(fp/(fp+tn)),fnr=float(fn/(fn+tp)),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))


In [ ]:
# E1: independent reference baseline. All threshold selection uses training data.
start=time.perf_counter()
e1_imputer=SimpleImputer(strategy='median',keep_empty_features=True).fit(X_full)
e1_scaler=MinMaxScaler().fit(e1_imputer.transform(X_full))
train_if=e1_scaler.transform(e1_imputer.transform(X_full))
e1_forest=CustomIsolationForest(n_trees=100,sample_size=256,random_state=RANDOM_STATE).fit(train_if)
e1_fit_seconds=time.perf_counter()-start
start=time.perf_counter(); train_scores=e1_forest.anomaly_score(train_if)
sweep=[]
for contamination in np.arange(.10,.401,.05):
    threshold=float(np.quantile(train_scores,1-contamination))
    pred=(train_scores<=threshold).astype(int)
    sweep.append(dict(contamination=float(contamination),threshold=threshold,
        train_accuracy=float(accuracy_score(y_full,pred))))
sweep=pd.DataFrame(sweep)
best=sweep.loc[sweep.train_accuracy.idxmax()]
e1_selection_seconds=time.perf_counter()-start
display(sweep)
print('0.25 is the published reference; selected contamination:',best.contamination)
start=time.perf_counter()
e1_scores=e1_forest.anomaly_score(e1_scaler.transform(e1_imputer.transform(X_test)))
e1_pred=(e1_scores<=best.threshold).astype(int)
e1_inference_seconds=time.perf_counter()-start
models={}; thresholds={'E1':float(best.threshold)}
predictions={'E1':e1_pred}; score_arrays={'E1':-e1_scores}
rows=[dict(experiment='E1',description='IF reference',features=9,smote=False,
    training_seconds=e1_fit_seconds,selection_seconds=e1_selection_seconds,
    inference_seconds=e1_inference_seconds,**evaluate(y_test,e1_pred,-e1_scores))]


In [ ]:
# E2–E5: fold-local preprocessing and resampling; same fit rows and CV splits.
cv=list(StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_STATE).split(X_fit,y_fit))
if min(np.bincount(y_fit)) < 10:
    raise ValueError('Too few minority examples for 5-fold CV and SMOTE(k=5)')
configs={'E2':(False,False),'E3':(False,True),'E4':(True,False),'E5':(True,True)}
search_metadata={}
for name,(augmented,smote) in configs.items():
    print('Starting',name,'augmentation=',augmented,'SMOTE=',smote)
    start=time.perf_counter()
    search=GridSearchCV(make_model(augmented,smote),PARAM_GRID,scoring=CV_SCORING,
        cv=cv,n_jobs=N_JOBS,refit=True,error_score='raise',verbose=1)
    search.fit(X_fit,y_fit)
    training_seconds=time.perf_counter()-start
    model=search.best_estimator_
    start=time.perf_counter()
    val_scores=model.decision_function(X_val)
    threshold,validation_f05=select_threshold(y_val,val_scores)
    selection_seconds=time.perf_counter()-start
    start=time.perf_counter()
    test_scores=model.decision_function(X_test)  # SMOTE is bypassed at inference
    pred=(test_scores>=threshold).astype(int)
    inference_seconds=time.perf_counter()-start
    models[name]=model; thresholds[name]=threshold
    predictions[name]=pred; score_arrays[name]=test_scores
    search_metadata[name]=dict(best_params=search.best_params_,best_cv_score=float(search.best_score_),
                              threshold=threshold,validation_f0_5=validation_f05)
    rows.append(dict(experiment=name,description='RBF-SVM'+(' + IF score' if augmented else '')+(' + SMOTE' if smote else ''),
        features=10 if augmented else 9,smote=smote,training_seconds=training_seconds,
        selection_seconds=selection_seconds,inference_seconds=inference_seconds,
        **evaluate(y_test,pred,test_scores)))
    pd.DataFrame(search.cv_results_).to_csv(OUTPUT/f'{name}_cv_results.csv',index=False)
comparison=pd.DataFrame(rows).set_index('experiment')
display(comparison)
print('Training seconds for E2–E5 include CV search plus refit; selection seconds reported separately.')


In [ ]:
# Paired exact McNemar test: E2 vs E4, as specified in Section 2.9.
correct_e2=predictions['E2']==y_test; correct_e4=predictions['E4']==y_test
b=int(np.sum(correct_e2 & ~correct_e4))
c=int(np.sum(~correct_e2 & correct_e4))
pvalue=float(binomtest(b,b+c,p=.5,alternative='two-sided').pvalue) if b+c else 1.0
mcnemar=dict(comparison='E2 vs E4',method='exact two-sided binomial McNemar',
    e2_only_correct=b,e4_only_correct=c,discordant=b+c,pvalue=pvalue,alpha=.05,
    reject_null=bool(pvalue<.05))
print(json.dumps(mcnemar,indent=2))
effects=[]
for left,right,purpose in [('E1','E5','Primary full architecture comparison'),
    ('E2','E4','IF score without SMOTE'),('E3','E5','IF score with SMOTE'),
    ('E2','E3','SMOTE without IF score'),('E4','E5','SMOTE with IF score')]:
    effects.append(dict(comparison=f'{right} minus {left}',purpose=purpose,
        **{metric:float(comparison.loc[right,metric]-comparison.loc[left,metric])
           for metric in ['accuracy','precision','recall','f1','f0_5','mcc']}))
effects=pd.DataFrame(effects);display(effects)


In [ ]:
fig,axes=plt.subplots(1,5,figsize=(19,4))
for ax,name in zip(axes,predictions):
    cm=confusion_matrix(y_test,predictions[name],labels=[0,1])
    ax.imshow(cm,cmap='Blues')
    for (i,j),value in np.ndenumerate(cm):ax.text(j,i,str(value),ha='center',va='center')
    ax.set(title=name,xlabel='Predicted',ylabel='Actual',xticks=[0,1],yticks=[0,1],
           xticklabels=['Normal','Attack'],yticklabels=['Normal','Attack'])
fig.tight_layout();fig.savefig(OUTPUT/'confusion_matrices.png',dpi=180);plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
for name,scores in score_arrays.items():
    fpr,tpr,_=roc_curve(y_test,scores)
    precision,recall,_=precision_recall_curve(y_test,scores)
    axes[0].plot(fpr,tpr,label=name);axes[1].plot(recall,precision,label=name)
axes[0].set(xlabel='False-positive rate',ylabel='Recall',title='ROC curves')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision–recall curves')
for ax in axes:ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUTPUT/'roc_pr_curves.png',dpi=180);plt.show()


In [ ]:
# Inspect the final proposed E5 inputs: all nine values plus normalized IF score.
def inspect_record(index=0):
    if index<0 or index>=len(X_test):raise IndexError('Invalid zero-based test row')
    record=X_test.iloc[[index]]
    model=models['E5']
    clean=model.named_steps['imputer'].transform(record)
    augmented=model.named_steps['augment'].transform(clean)
    standardized=model.named_steps['standardize'].transform(augmented)
    display(pd.DataFrame(dict(feature=FEATURES,
        observed=record.iloc[0].to_numpy(),imputed_input=clean[0],
        was_missing=record.iloc[0].isna().to_numpy())))
    print('Normalized IF score:',float(augmented[0,-1]),'(not a probability)')
    display(pd.DataFrame(dict(input=FEATURES+['normalized_if_score'],standardized=standardized[0])))
    print('Predictions:',{k:int(v[index]) for k,v in predictions.items()},'Actual:',int(y_test[index]))
    print('Input inspection is not feature attribution or a causal explanation.')
inspect_record(0)


In [ ]:
def sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()
metadata=dict(architecture='E1-E5 corrected paper; score normalization + optional training-only SMOTE',
    random_state=RANDOM_STATE,validation_size=VALIDATION_SIZE,svm_fit_rows=len(y_fit),
    cv_folds=CV_FOLDS,cv_scoring=CV_SCORING,param_grid=PARAM_GRID,
    python=platform.python_version(),sklearn=sklearn.__version__,imblearn=imblearn.__version__,
    numpy=np.__version__,pandas=pd.__version__,counts=counts.to_dict('records'),
    train_sha256=sha256(TRAIN_PATH),test_sha256=sha256(TEST_PATH))
comparison.to_csv(OUTPUT/'E1_E5_comparison.csv')
effects.to_csv(OUTPUT/'ablation_effects.csv',index=False)
sweep.to_csv(OUTPUT/'IF_contamination_sweep.csv',index=False)
pd.DataFrame(dict(actual=y_test,**predictions)).to_csv(OUTPUT/'test_predictions.csv',index=False)
Path(OUTPUT/'experiment_evidence.json').write_text(json.dumps(dict(metadata=metadata,
    tuning=search_metadata,mcnemar=mcnemar,results=comparison.reset_index().to_dict('records')),
    indent=2,allow_nan=False))
joblib.dump(dict(metadata=metadata,features=FEATURES,models=models,thresholds=thresholds,
    e1_imputer=e1_imputer,e1_scaler=e1_scaler,e1_forest=e1_forest),OUTPUT/'E1_E5_models.joblib')
print('Saved artifacts to',OUTPUT.resolve())
print('Bundle schema changed: the earlier Laravel worker cannot read this bundle without an adapter.')
print('Custom notebook classes must be available when loading joblib; use only trusted model files.')


## Reading the results
Report actual E1–E5 values after execution. Interpret improved recall alongside false alarms, precision, MCC and timing; do not presume E5 wins. Statistical significance from McNemar is separate from the magnitude of improvement. All timing measurements come from this execution.

The score normalization, 80/20 split, SMOTE configurations and gamma grid differ from the previous notebook. Earlier three-model outputs are therefore not valid substitutes for the new experiment.

Implementation references: [imbalanced-learn pipeline](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html), [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html), [SciPy exact binomial test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html). The corrected uploaded manuscript is the experiment specification.
